# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [2]:
import asyncio
from dotenv import load_dotenv
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
load_dotenv()
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [3]:
DATA_DIR = Path('data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [4]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""

    return [
        {
            "role": "user",
            "content": f"""
                Extract the following fields from this job posting:
                    - company
                    - role
                    - years_experience_required

                Rules:
                    - Return company, role, and years_experience_required.
                    - Return the minimum value for experience ranges.
                    - Convert written numbers to integers.
                    - Return 0 only when no experience is required.
                    - Return null when no specific experience requirement is stated.
                    - Do not infer or invent missing information.
                    - Return only one valid JSON object.
                
                Job Posting: {snippet_text} 
            """.strip()
        }
    ]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""

    examples = """
    ### Example 1

    Job Posting:
    We are hiring a Data Analyst at Northwind Ltd. Preferred 2 years of experience.

    Expected JSON:
    {
        "company": "Northwind Ltd",
        "role": "Data Analyst",
        "years_experience_required": 2
    }


    ### Example 2

    Job Posting:
    Acme Corp is looking for a Senior Software Engineer with at least 5 years of experience.

    Expected JSON:
    {
        "company": "Acme Corp",
        "role": "Senior Software Engineer",
        "years_experience_required": 5
    }


    ### Example 3

    Job Posting:
    Hooli is looking for a Junior Frontend Developer. No prior experience required.

    Rules:
        - Return company, role, and years_experience_required.
        - Return the minimum value for experience ranges.
        - Convert written numbers to integers.
        - Return 0 only when no experience is required.
        - Return null when no specific experience requirement is stated.
        - Do not infer or invent missing information.
        - Return only one valid JSON object.

    Expected JSON:
    {
        "company": "Hooli",
        "role": "Junior Frontend Developer",
        "years_experience_required": 0
    }
    """.strip()


    prompt = f"""
    You are an information extraction system.

    Your task is to extract structured information from a job posting.

    ### Required Output Fields

    - company
    - role
    - years_experience_required

    ### Few-Shot Examples

    {examples}

    ### Target Job Posting

    {snippet_text}

    ### Expected Response

    Return only the JSON object.

    """.strip()


    return [
        {
            "role": "user",
            "content": prompt,
        }
    ]   


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    system_prompt = """
    You are an expert job-posting information extraction system.

    Your task is to extract structured information from job postings.

    ## Output Schema

    Return exactly one JSON object with these fields:

    {
        "company": "string",
        "role": "string",
        "years_experience_required": "integer | null"
    }

    ## Extraction Rules

    1. Return only one valid JSON object.
    2. Use exactly these field names:
        - company
        - role
        - years_experience_required
    3. Do not include Markdown, explanations, comments, or additional fields.
    4. Extract the company name exactly as stated in the job posting.
    5. Extract the job role exactly as stated in the job posting.
    6. If an experience range is specified, return the minimum number.
        Example: "3-5 years" → 3
    7. Convert written numbers to integers.
        Example: "five years" → 5
    8. If the posting explicitly says no experience is required, return 0.
    9. If no specific years-of-experience requirement is stated, return null.
    10. Do not infer, assume, or invent missing information.

    ## Example

    Input:
    "We are hiring a Data Analyst at Northwind Ltd. Preferred 2 years of experience."

    Output:
    {
        "company": "Northwind Ltd",
        "role": "Data Analyst",
        "years_experience_required": 2
    }
    """.strip()

    user_prompt = f"""
    Extract the requested information from the following job posting.

    ### Job Posting

    {snippet_text}

    Return only the JSON object.
    """.strip()

    return [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]   


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — Reasoning-based prompting.

    Ask the model to carefully analyze the input before producing
    the final structured JSON response.
    """

    return [
        {
            "role": "system",
            "content": """
                You are an expert job-posting information extraction system.

                Analyze the job posting carefully before producing the final answer.

                Determine:
                    - company
                    - role
                    - minimum required years of experience

                Rules:
                    - Return company, role, and years_experience_required.
                    - Return the minimum value for experience ranges.
                    - Convert written numbers to integers.
                    - Return 0 only when no experience is required.
                    - Return null when no specific experience requirement is stated.
                    - Do not infer or invent missing information.
                    - Return only one valid JSON object.

                Think through the information carefully, then return only the final JSON.
                """.strip(),
        },
        {
            "role": "user",
            "content": f"""
                Job Posting:
                {snippet_text}
                """.strip(),
        },
]       


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [5]:
def parse_response(text: str) -> dict | None:
    """Parse a JSON object from the model response.

    Handles plain JSON and JSON wrapped in Markdown code fences.
    Returns None when parsing fails.
    """
    
    if not isinstance(text, str):
        return None

    cleaned = text.strip()

    # Remove Markdown fences such as ```json ... ```
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()

        # Remove the opening fence and optional language name.
        lines = lines[1:]

        # Remove the closing fence.
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        cleaned = "\n".join(lines).strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        return None

    return parsed if isinstance(parsed, dict) else None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet and capture response, cost, and latency."""

    start_time = time.perf_counter()

    try:
        messages = STRATEGIES[strategy_name](snippet["snippet"])

        response = await client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=TEMPERATURE,
        )

        raw_response = response.choices[0].message.content or ""
        usage = response.usage

        prompt_tokens = usage.prompt_tokens if usage else 0
        completion_tokens = usage.completion_tokens if usage else 0

        cost_usd = (
            prompt_tokens * RATES[MODEL]["in"]
            + completion_tokens * RATES[MODEL]["out"]
        )

        extracted = parse_response(raw_response)

        return {
            "strategy": strategy_name,
            "snippet_id": snippet["id"],
            "raw_response": raw_response,
            "extracted": extracted,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "cost_usd": round(cost_usd, 12),
            "latency_s": round(time.perf_counter() - start_time, 6),
        }

    except Exception as exc:
        return {
            "strategy": strategy_name,
            "snippet_id": snippet.get("id"),
            "raw_response": "",
            "extracted": None,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "cost_usd": 0.0,
            "latency_s": time.perf_counter() - start_time,
            "error": str(exc),
        }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    
    tasks = [
        run_one(strategy_name, snippet)
        for strategy_name in STRATEGIES.keys()
        for snippet in snippets
    ]

    return await asyncio.gather(*tasks)

In [6]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '```json\n{\n    "company": "Acme Corp",\n    "role": "Senior Software Engineer",\n    "years_experience_required": 5\n}\n```',
 'extracted': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'prompt_tokens': 148,
 'completion_tokens': 34,
 'cost_usd': 4.26e-05,
 'latency_s': 1.949182}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [7]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare the three fields and return a score from 0 to 3."""

    if not isinstance(extracted, dict):
        return 0

    fields = [
        "company",
        "role",
        "years_experience_required",
    ]

    score = 0

    for field in fields:
        predicted = extracted.get(field)
        expected = gold.get(field)

        if isinstance(predicted, str) and isinstance(expected, str):
            matches = predicted.strip().casefold() == expected.strip().casefold()
        else:
            matches = predicted == expected

        if matches:
            score += 1

    return score


async def score_llm_judge(
    snippet_text: str,
    extracted: dict | None,
    gold: dict,
) -> int:
    """Use the judge model to score the extraction from 1 to 4."""

    judge_prompt = f"""
      Evaluate the extracted information from this job posting.

      Job posting:
      {snippet_text}

      Expected answer:
      {json.dumps(gold, ensure_ascii=False)}

      Model extraction:
      {json.dumps(extracted, ensure_ascii=False)}

      Use this rubric:
        - 4: all three fields are correct
        - 3: two of three fields are correct and no information was fabricated
        - 2: one field is correct, or the answer contains fabricated information
        - 1: none of the fields are correct or the extraction is unparsable

      Return only one valid JSON object in this format:
      {{"score": 1}}

      The score must be an integer from 1 to 4.
      """.strip()

    try:
        response = await client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a strict evaluator of structured job-posting extractions."
                    ),
                },
                {
                    "role": "user",
                    "content": judge_prompt,
                },
            ],
            temperature=0.0,
        )

        judge_text = response.choices[0].message.content or ""
        judge_result = parse_response(judge_text)

        if not isinstance(judge_result, dict):
            return 1

        score = judge_result.get("score")

        if isinstance(score, int) and 1 <= score <= 4:
            return score

        return 1

    except Exception:
        return 1

In [8]:
# Apply scoring to all 40 results

snippet_text_by_id = {
    item["id"]: item["snippet"]
    for item in snippets
}

async def score_one(result: dict) -> dict:
    """Attach deterministic and LLM-based scores to one result."""
    snippet_id = result["snippet_id"]
    extracted = result.get("extracted")
    gold_entry = golden[snippet_id]

    scored_result = dict(result)
    scored_result["accuracy"] = score_accuracy(extracted, gold_entry)
    scored_result["parse_success"] = extracted is not None
    scored_result["llm_judge_score"] = await score_llm_judge(
        snippet_text_by_id[snippet_id],
        extracted,
        gold_entry,
    )

    return scored_result


scored = await asyncio.gather(
    *(score_one(result) for result in results)
)

print(f"Scored {len(scored)} results.")

Scored 40 results.


## Step 5 — Build the comparison table

In [9]:
df = pd.DataFrame(scored)

strategy_order = list(STRATEGIES.keys())

summary = (
    df.groupby("strategy", sort=False)
    .agg(
        accuracy=("accuracy", "mean"),
        parse_success=("parse_success", "mean"),
        llm_judge_score=("llm_judge_score", "mean"),
        cost_usd=("cost_usd", "sum"),
        latency_s=("latency_s", "median"),
    )
    .reindex(strategy_order)
)

# Convert the 1–4 judge score to a 25-point scale.
summary["llm_judge_score"] = summary["llm_judge_score"] * (25 / 4)

summary = summary.round(
    {
        "accuracy": 2,
        "parse_success": 4,
        "llm_judge_score": 2,
        "cost_usd": 8,
        "latency_s": 3,
    }
)

summary.columns = [
    "Accuracy (mean of 3)",
    "Parse rate",
    "Judge score (out of 25)",
    "Total cost ($)",
    "Latency p50 (s)",
]

summary.style.format(
    {
        "Accuracy (mean of 3)": "{:.2f}",
        "Parse rate": "{:.2%}",
        "Judge score (out of 25)": "{:.2f}",
        "Total cost ($)": "${:.8f}",
        "Latency p50 (s)": "{:.3f}",
    }
)

,Accuracy (mean of 3),Parse rate,Judge score (out of 25),Total cost ($),Latency p50 (s)
strategy,,,,,
zero_shot,2.90,100.00%,20.00,$0.00043470,1.824
few_shot,3.00,100.00%,19.38,$0.00075870,1.929
structured,3.00,100.00%,19.38,$0.00077370,1.673
cot,2.80,100.00%,20.00,$0.00046470,1.741


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```